# NB151: Dynamics First — Does the Simulation Produce the Standard Model?

**The test**: Start from ONLY the single action principle — gradient flow of V_covering with Γ̃ = K·A⁻¹ dissipation, κ = 1/√P₄, ω = 2π — and the solenoid initial conditions R_k(0) = 2πj_{k+1}. Integrate all 210 branches. Evaluate at all 48 coprime crossings. Extract the FULL set of pairwise ratios. See if the Standard Model mass spectrum emerges WITHOUT any algebraic labels or assignments imposed from outside.

**What we provide**: The four primes {2,3,5,7} and nothing else. The cascade ODE follows from the primes (NB139, NB143). The initial conditions ARE the solenoid sheets. The coprime crossings ARE the integers coprime to 210.

**What we DON'T provide**: CP pair assignments, CRT quantum number labels, mass exponents, fermion identities, or any SM input. The simulation doesn't know about quarks, leptons, generations, or color.

**The success criterion**: The PDG mass ratios (m_μ/m_e = 206.77, m_s/m_d = 20.0, m_τ/m_μ = 16.82, etc.) appear as distinguished pairwise ratios in the dynamical output — ratios that stand out from the background of all possible pairwise comparisons.

**Why this matters**: If the masses emerge from the dynamics alone, the algebra is a READING of the dynamics, not an independent input. The fermion assignments are determined by the dynamics, not imposed on it. The selection mechanism is the gradient flow itself.

## S0: The Raw Dynamics — 48 Sector RMS Values From the Cascade

Step 1: integrate the cascade and compute the sector RMS at each of the 48 coprime crossings. No CRT labels, no CP pairs — just 48 numbers from the dynamics.

Each sector RMS is computed as: RMS(ci) = √((1/210) Σ_branches wrap(R₃(ci; branch))²)

This is the full branch-averaged wrapped squared residual at level 3 (the mass level) for each coprime crossing time. These 48 numbers are the COMPLETE dynamical output relevant to mass. Everything else should follow from them.

In [3]:
# ── S0: The raw dynamics — 48 sector RMS values ──

import sys, os, time, numpy as np
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, KAPPA, EPSILON, OMEGA
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

print('S0: THE RAW DYNAMICS — 48 NUMBERS FROM THE CASCADE')
print('='*70)

# ONLY INPUTS: the four primes
primes = [2, 3, 5, 7]
P4 = 2 * 3 * 5 * 7  # = 210
print(f'Input: primes = {primes}, P4 = {P4}')

# DERIVED from primes (NB139, NB143):
kappa = 1 / np.sqrt(P4)
omega = 2 * np.pi
print(f'Derived: kappa = 1/sqrt({P4}) = {kappa:.6f}, omega = 2*pi')

# The coprime crossing times: integers in [1, P4) coprime to P4
from math import gcd
cis = np.array(sorted([ci for ci in range(1, P4) if gcd(ci, P4) == 1]))
print(f'Coprime crossings: {len(cis)} in [1, {P4})')

# Integrate the cascade
sys0 = SolenoidSystem()
all_branches = sys0.all_branches()

print(f'\nJAX device: {detect_device()}')
t0 = time.time()
jax_warmup()
print(f'JAX warmup: {time.time()-t0:.1f}s')

t_eval = cis.astype(float)
T_max = float(P4 + 1)

t0 = time.time()
res = sys0.integrate_all_branches(all_branches, t_eval, T_max, backend='jax')
dt = time.time() - t0
print(f'Integration: {dt:.1f}s')

# Compute sector RMS at R3 for each crossing (NO CRT labels, just the numbers)
rms_R3 = np.zeros(len(cis))
for idx in range(len(cis)):
    R3_all = np.array([res[br][idx, 3] for br in all_branches])
    R3_wrapped = np.mod(R3_all, 2*np.pi)
    R3_wrapped[R3_wrapped > np.pi] -= 2*np.pi
    rms_R3[idx] = np.sqrt(np.mean(R3_wrapped**2))

# Also compute for all 4 levels
rms_all_levels = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk_all = np.array([res[br][idx, k] for br in all_branches])
        Rk_wrapped = np.mod(Rk_all, 2*np.pi)
        Rk_wrapped[Rk_wrapped > np.pi] -= 2*np.pi
        rms_all_levels[idx, k] = np.sqrt(np.mean(Rk_wrapped**2))

# Display: just the 48 numbers, sorted by crossing time
print(f'\nTHE 48 NUMBERS (R3 sector RMS, sorted by crossing time):')
print(f'  {"ci":>4s}  {"RMS(R3)":>10s}  {"RMS(R0)":>10s}  {"RMS(R1)":>10s}  {"RMS(R2)":>10s}')
for idx in range(len(cis)):
    ci = cis[idx]
    print(f'  {ci:4d}  {rms_R3[idx]:10.6f}  '
          f'{rms_all_levels[idx,0]:10.6f}  {rms_all_levels[idx,1]:10.6f}  {rms_all_levels[idx,2]:10.6f}')

# Summary statistics
print(f'\nSummary of R3 sector RMS across 48 crossings:')
print(f'  min = {rms_R3.min():.6f} at ci = {cis[np.argmin(rms_R3)]}')
print(f'  max = {rms_R3.max():.6f} at ci = {cis[np.argmax(rms_R3)]}')
print(f'  mean = {rms_R3.mean():.6f}')
print(f'  std = {rms_R3.std():.6f}')
print(f'  max/min = {rms_R3.max()/rms_R3.min():.4f}')

# The 48 numbers span from ~0.25 to ~2.08. 
# Now we'll search for mass ratios among ALL pairwise comparisons.

S0: THE RAW DYNAMICS — 48 NUMBERS FROM THE CASCADE
Input: primes = [2, 3, 5, 7], P4 = 210
Derived: kappa = 1/sqrt(210) = 0.069007, omega = 2*pi
Coprime crossings: 48 in [1, 210)

JAX device: CPU (1 device(s))
JAX warmup: 1.2s
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.29s
Integration: 2.3s

THE 48 NUMBERS (R3 sector RMS, sorted by crossing time):
    ci     RMS(R3)     RMS(R0)     RMS(R1)     RMS(R2)
     1    1.380226    0.296767    0.470840    0.921373
    11    1.846494    2.075590    1.618601    1.737103
    13    1.884559    1.807021    1.630157    1.801197
    17    1.712628    1.369288    1.912233    1.811659
    19    1.843493    1.191773    1.979964    1.711459
    23    1.714366    0.902447    1.921001    1.789444
    29    1.862250    0.593886    1.533449    2.075967
    31    1.973601    0.516338    1.367818    2.089689
    37    2.166001    0.338700    0.970860    1.702667
    41    2.076132    0.255177    0.772792    1.308738
    43    1.912028    0

## S1: Blind Search — Do Mass Ratios Appear in the Pairwise Comparisons?

We have 48 sector RMS values. There are C(48,2) = 1128 ordered pairs (i,j) with RMS(i) > RMS(j). For each pair, the ratio RMS(i)/RMS(j) is a candidate "CP ratio." We need to check whether ANY of the known mass ratios appear when raised to plausible exponents.

The mass formula is: mass_ratio = CP^x, so CP = mass_ratio^(1/x).

Known mass ratios and their CP values at known exponents:
- m_μ/m_e = 206.77, with x = 3 → CP = 206.77^(1/3) = 5.914
- m_s/m_d = 20.0, with x = 100/63 → CP = 20.0^(63/100) = 9.03... wait, that uses the exponent.

The blind test should be: find pairs where RMS(i)/RMS(j) is close to known CP values. But this requires knowing the CP values, which requires knowing the exponents.

A truly blind test: compute RMS(i)/RMS(j) for all 1128 pairs, and check if ANY of them, raised to ANY small integer power, give a known mass ratio. Let me compute CP^n for n = 1,...,10 for each pair and see which match PDG ratios.

In [5]:
# ── S1: Blind search for mass ratios in pairwise RMS comparisons ──

print('S1: BLIND SEARCH — MASS RATIOS IN PAIRWISE COMPARISONS')
print('='*70)

# Known mass ratios (PDG values, from SM_TARGETS)
mass_ratios = {
    'm_mu/m_e': 206.768,
    'm_s/m_d': 20.0,
    'm_tau/m_mu': 16.817,
    'm_c/m_s': 11.72,
    'm_b/m_tau': 2.356,   # m_b/m_tau ≈ 4.18/1.777
    'm_t/m_b': 42.0,      # P4/p3
    'm_t/M_Z': 1.92,      # 175/91.2
}

# Compute ALL pairwise RMS ratios at R3
# Use the SORTED RMS values but keep track of which ci they came from
n = len(cis)
all_ratios = []
for i in range(n):
    for j in range(n):
        if i != j and rms_R3[i] > rms_R3[j] and rms_R3[j] > 0.01:
            ratio = rms_R3[i] / rms_R3[j]
            all_ratios.append((ratio, cis[i], cis[j]))

all_ratios.sort(key=lambda x: x[0])
print(f'Total pairwise ratios with RMS > 0.01: {len(all_ratios)}')
print(f'Ratio range: [{all_ratios[0][0]:.4f}, {all_ratios[-1][0]:.4f}]')

# For each mass ratio, search for pairs where CP^n ≈ mass_ratio
# for small integer and half-integer n
print(f'\nBLIND SEARCH: CP^n ≈ mass_ratio for n = 1, 1.5, 2, 2.5, 3, 4, 5')
print(f'Tolerance: 5% relative deviation')

exponents_to_try = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
tolerance = 0.05  # 5% relative

for mass_name, mass_target in mass_ratios.items():
    print(f'\n  {mass_name} = {mass_target}:')
    found_any = False
    
    for x in exponents_to_try:
        # CP needed: mass_target^(1/x)
        cp_needed = mass_target ** (1.0/x)
        
        # Search for this CP in all_ratios
        matches = []
        for ratio, ci_i, ci_j in all_ratios:
            if abs(ratio - cp_needed) / cp_needed < tolerance:
                dev_pct = 100 * (ratio - cp_needed) / cp_needed
                matches.append((ratio, ci_i, ci_j, dev_pct))
        
        if matches:
            found_any = True
            for ratio, ci_i, ci_j, dev in matches[:3]:  # show top 3
                print(f'    x={x:.1f}: CP needed={cp_needed:.4f}, '
                      f'found CP={ratio:.4f} at ci={ci_i}/{ci_j} ({dev:+.2f}%)')
    
    if not found_any:
        print(f'    NO matches found within 5% for any exponent')

# Now: the STRONGEST test. Among all (CP, x) combinations, which ones
# give mass ratios closest to PDG? Plot deviation vs exponent for the 
# physical CP pairs (which the simulation doesn't know about).

print(f'\n\n{"="*70}')
print(f'REVERSE TEST: For the KNOWN CP pairs, what mass ratios emerge?')
print(f'{"="*70}')

# The CP pairs are at crossings (ci=11, ci=191) and (ci=31, ci=61)
# Let me find these without using CRT labels — just by their ci values
known_pairs = [
    ('QUARK', 11, 191),
    ('LEPTON', 31, 61),
]

for name, ci_g1, ci_g2 in known_pairs:
    idx_g1 = np.where(cis == ci_g1)[0][0]
    idx_g2 = np.where(cis == ci_g2)[0][0]
    cp = rms_R3[idx_g1] / rms_R3[idx_g2]
    
    print(f'\n  {name}: ci={ci_g1}/{ci_g2}, CP = {cp:.6f}')
    print(f'    CP^n for various n:')
    for x in [1.0, 1.5, 2.0, 2.5, 3.0, 100/63, 12/(2*np.pi)]:
        mass_pred = cp ** x
        # Find closest PDG match
        best_match = min(mass_ratios.items(), key=lambda mr: abs(np.log(mass_pred/mr[1])))
        dev = 100 * (mass_pred - best_match[1]) / best_match[1]
        x_label = f'{x:.4f}' if x not in [1,1.5,2,2.5,3] else f'{x:.1f}'
        print(f'      x = {x_label}: CP^x = {mass_pred:.4f}, '
              f'closest SM: {best_match[0]} = {best_match[1]:.3f} ({dev:+.2f}%)')

S1: BLIND SEARCH — MASS RATIOS IN PAIRWISE COMPARISONS
Total pairwise ratios with RMS > 0.01: 1128
Ratio range: [1.0010, 39.1019]

BLIND SEARCH: CP^n ≈ mass_ratio for n = 1, 1.5, 2, 2.5, 3, 4, 5
Tolerance: 5% relative deviation

  m_mu/m_e = 206.768:
    x=1.5: CP needed=34.9668, found CP=33.2798 at ci=19/83 (-4.82%)
    x=1.5: CP needed=34.9668, found CP=33.3340 at ci=11/83 (-4.67%)
    x=1.5: CP needed=34.9668, found CP=33.6184 at ci=29/83 (-3.86%)
    x=2.0: CP needed=14.3794, found CP=13.7571 at ci=43/97 (-4.33%)
    x=2.0: CP needed=14.3794, found CP=13.7868 at ci=13/139 (-4.12%)
    x=2.0: CP needed=14.3794, found CP=13.9138 at ci=19/169 (-3.24%)
    x=2.5: CP needed=8.4371, found CP=8.0631 at ci=31/137 (-4.43%)
    x=2.5: CP needed=8.4371, found CP=8.1284 at ci=73/187 (-3.66%)
    x=2.5: CP needed=8.4371, found CP=8.1523 at ci=37/181 (-3.38%)
    x=3.0: CP needed=5.9133, found CP=5.6290 at ci=29/101 (-4.81%)
    x=3.0: CP needed=5.9133, found CP=5.6389 at ci=1/137 (-4.64%)
    x

## S2: Tightening the Search — Do the Physical Pairs Emerge at Sub-Percent Precision?

The 5% blind search found hundreds of near-matches. But the physical CP pairs give mass ratios to better than 1%. Let me tighten the tolerance to 1% and 0.1% and see how many pairs survive — if only the physical CP pairs survive at high precision, the dynamics IS selecting them.

I'll also check: for the physical exponents (x = 3 for lepton, x = 100/63 for quark), how many DISTINCT CP pairs give the correct mass ratio within 1%? If it's only one pair per channel, the selection is unique.

In [7]:
# ── S2: Precision search — do the physical pairs emerge? ──

print('S2: PRECISION SEARCH — SUB-PERCENT TOLERANCE')
print('='*70)

# For each (mass_ratio, exponent) pair, count how many CP pairs match
# at various tolerance levels

test_cases = [
    ('m_mu/m_e', 206.768, 3.0, 31, 61),
    ('m_s/m_d', 20.0, 100/63, 11, 191),
    ('m_tau/m_mu', 16.817, 100/63, 31, 61),
    ('m_t/m_b', 42.0, 2.0, 11, 191),
]

for mass_name, mass_target, x, phys_g1, phys_g2 in test_cases:
    cp_needed = mass_target ** (1.0/x)
    
    print(f'\n  {mass_name} = {mass_target}, x = {x:.4f}, CP needed = {cp_needed:.4f}')
    print(f'  Physical pair: ci={phys_g1}/{phys_g2}')
    
    for tol in [0.05, 0.01, 0.005, 0.001, 0.0005]:
        matches = []
        phys_found = False
        for ratio, ci_i, ci_j in all_ratios:
            if abs(ratio - cp_needed) / cp_needed < tol:
                dev_pct = 100 * (ratio - cp_needed) / cp_needed
                matches.append((ratio, int(ci_i), int(ci_j), dev_pct))
                if (int(ci_i) == phys_g1 and int(ci_j) == phys_g2) or \
                   (int(ci_i) == phys_g2 and int(ci_j) == phys_g1):
                    phys_found = True
        
        n = len(matches)
        phys_marker = ' ✓ PHYS' if phys_found else ' ✗ no phys'
        if n > 0 and n <= 5:
            detail = ', '.join(f'{ci_i}/{ci_j}' for _, ci_i, ci_j, _ in matches)
            print(f'    {tol*100:6.2f}% tol: {n:3d} matches [{detail}]{phys_marker}')
        elif n > 5:
            print(f'    {tol*100:6.2f}% tol: {n:3d} matches{phys_marker}')
        else:
            print(f'    {tol*100:6.2f}% tol:   0 matches')

# Multi-level test
print(f'\n\n{"="*70}')
print(f'MULTI-LEVEL TEST: Same crossing pair, different cascade levels')
print(f'{"="*70}')

for name, ci_g1, ci_g2 in [('QUARK', 11, 191), ('LEPTON', 31, 61)]:
    idx_g1 = np.where(cis == ci_g1)[0][0]
    idx_g2 = np.where(cis == ci_g2)[0][0]
    
    print(f'\n  {name} pair (ci={ci_g1}/{ci_g2}):')
    for k in range(4):
        cp_k = rms_all_levels[idx_g1, k] / rms_all_levels[idx_g2, k]
        matches_at_level = []
        for x in [1.0, 1.5, 2.0, 3.0, 100/63, 12/(2*np.pi)]:
            mass_pred = cp_k ** x
            for mr_name, mr_val in mass_ratios.items():
                dev = 100 * (mass_pred - mr_val) / mr_val
                if abs(dev) < 5:
                    matches_at_level.append((k, x, mass_pred, mr_name, mr_val, dev))
        
        print(f'    R{k}: CP = {cp_k:.6f}')
        for k2, x, mp, mn, mv, dev in matches_at_level:
            print(f'      x={x:.4f}: {mp:.4f} ≈ {mn} = {mv} ({dev:+.2f}%)')

S2: PRECISION SEARCH — SUB-PERCENT TOLERANCE

  m_mu/m_e = 206.768, x = 3.0000, CP needed = 5.9133
  Physical pair: ci=31/61
      5.00% tol:  59 matches ✓ PHYS
      1.00% tol:  11 matches ✓ PHYS
      0.50% tol:   5 matches [31/61, 13/163, 47/137, 13/193, 73/113] ✓ PHYS
      0.10% tol:   2 matches [31/61, 13/163] ✓ PHYS
      0.05% tol:   1 matches [31/61] ✓ PHYS

  m_s/m_d = 20.0, x = 1.5873, CP needed = 6.6016
  Physical pair: ci=11/191
      5.00% tol:  48 matches ✓ PHYS
      1.00% tol:  12 matches ✓ PHYS
      0.50% tol:   5 matches [31/149, 19/191, 11/191, 1/89, 67/187] ✓ PHYS
      0.10% tol:   2 matches [19/191, 11/191] ✓ PHYS
      0.05% tol:   0 matches

  m_tau/m_mu = 16.817, x = 1.5873, CP needed = 5.9186
  Physical pair: ci=31/61
      5.00% tol:  60 matches ✓ PHYS
      1.00% tol:  12 matches ✓ PHYS
      0.50% tol:   6 matches ✓ PHYS
      0.10% tol:   1 matches [13/163] ✗ no phys
      0.05% tol:   1 matches [13/163] ✗ no phys

  m_t/m_b = 42.0, x = 2.0000, CP needed